Import libraries

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

Load Multilingual-E5 SMALL model

In [ ]:
model_e5 = SentenceTransformer('intfloat/multilingual-e5-small')

Loading multilingual-E5-small model...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Model loaded!


Test with example data

In [3]:
document = "The court ruled on data privacy violations"
labels = ["data protection", "privacy law", "court ruling", "criminal law"]

print(f"Document: {document}")
print(f"Labels: {labels}")

Document: The court ruled on data privacy violations
Labels: ['data protection', 'privacy law', 'court ruling', 'criminal law']


Embed document and labels.

In [4]:
# Embed the document
doc_embedding = model_e5.encode(document)
print(f"Document embedding shape: {doc_embedding.shape}")

# Embed all labels
label_embeddings = model_e5.encode(labels)
print(f"Label embeddings shape: {label_embeddings.shape}")

Document embedding shape: (384,)
Label embeddings shape: (4, 384)


Calculate similarities

In [5]:
# Compute cosine similarity between document and each label
similarities = cosine_similarity([doc_embedding], label_embeddings)[0]

# Create a nice table
results_df = pd.DataFrame({
    'Label': labels,
    'Similarity': similarities
})
results_df = results_df.sort_values('Similarity', ascending=False)
print(results_df)

             Label  Similarity
2     court ruling    0.875486
1      privacy law    0.870575
0  data protection    0.868052
3     criminal law    0.829144


Retrieve top-k labels

In [6]:
k = 2
top_k_indices = np.argsort(similarities)[::-1][:k]
top_k_labels = [labels[i] for i in top_k_indices]
top_k_scores = [similarities[i] for i in top_k_indices]

print(f"\nTop {k} labels:")
for label, score in zip(top_k_labels, top_k_scores):
    print(f"  {label}: {score:.4f}")


Top 2 labels:
  court ruling: 0.8755
  privacy law: 0.8706


OpenAI embeddings

In [ ]:
# Only run if you have an API key and want to compare

client = OpenAI(api_key="your-api-key-here")  # Replace with your key

def get_openai_embedding(text):
    response = client.embeddings.create(
        input=text, 
        model="text-embedding-3-small"
    )
    return np.array(response.data[0].embedding)

# Embed with OpenAI
doc_embedding_oai = get_openai_embedding(document)
label_embeddings_oai = np.array([get_openai_embedding(label) for label in labels])

# Calculate similarities
similarities_oai = cosine_similarity([doc_embedding_oai], label_embeddings_oai)[0]

# Compare results
comparison_df = pd.DataFrame({
    'Label': labels,
    'E5 Similarity': similarities,
    'OpenAI Similarity': similarities_oai
})
comparison_df = comparison_df.sort_values('E5 Similarity', ascending=False)
print(comparison_df)